In [ ]:
%%writefile Dockerfile
FROM python:3.12-slim
WORKDIR /app
RUN pip install --no-cache-dir fastapi uvicorn joblib scikit-learn==1.8.0
COPY app.py digit_model.joblib ./
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]


In [ ]:
%%writefile .dockerignore
Datasets
Pretrained
*.ipynb


In [ ]:
!docker build -t digit-api .


In [ ]:
!docker run -d --name digit-api -p 8001:8000 digit-api


In [ ]:
import time
import numpy as np
import requests

test = np.loadtxt("Datasets/pendigits.tes", delimiter=",")
sample = {"coordinates": test[0, :16].astype(int).tolist()}
print("Request:", sample)

url = "http://127.0.0.1:8001/predict"
for i in range(30):
    try:
        r = requests.post(url, json=sample)
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

print("Status code:", r.status_code)
print("Response from container:", r.json())
print("Actual digit:", int(test[0, 16]))


In [ ]:
!docker stop digit-api
!docker rm digit-api
